In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import cross_validate, KFold

from catboost import CatBoostRegressor, cv, Pool

SEED = 1660

## Загружаем данные

In [2]:
df_train = pd.read_parquet("03_datasets/final_train.parquet")
df_test = pd.read_parquet("03_datasets/final_test.parquet")

In [3]:
X_train = df_train.drop(columns=["GameId", "Elo", "WhiteElo", "BlackElo"]).drop(columns=["LineTreeMean", "MeanStartLoss", "Opening"])
Y_train = df_train["Elo"]

X_test = df_test.drop(columns=["GameId", "Elo", "WhiteElo", "BlackElo"]).drop(columns=["LineTreeMean", "MeanStartLoss", "Opening"])
Y_test = df_test["Elo"]

In [4]:
X_train.shape, X_test.shape

((400000, 15), (100000, 15))

## Вспомогательные функции

In [5]:
def plot_bars(x, title):
    fig = px.bar(x)
    
    fig.data[0].marker.color="white"
    fig.data[1].marker.color="green"
    fig.data[0].marker.line.width=0
    fig.data[1].marker.line.width=0
    
    fig.update_layout(
        barmode="group", 
        bargroupgap=0.0,
        template="plotly_dark",
        xaxis_title="K",
        yaxis_title="Value",
        title_text=title
    )
    
    fig.show()

In [6]:
def get_crossval_report(model, X_train, Y_train):
    cv_scores = cross_validate(
        model, 
        X_train, Y_train, 
        cv=KFold(n_splits=3, random_state=SEED, shuffle=True),
        scoring=["r2", "neg_mean_absolute_error"],
        return_train_score=True
    )
    
    r2_scores = pd.DataFrame({
        "Train": cv_scores["train_r2"],
        "Test": cv_scores["test_r2"]
    })
    
    plot_bars(r2_scores*100, title="R^2")
    
    mae_scores = pd.DataFrame({
        "Train": cv_scores["train_neg_mean_absolute_error"],
        "Test": cv_scores["test_neg_mean_absolute_error"]
    })
    
    plot_bars(-1*mae_scores, title="MAE")

## Линейная модель

In [7]:
linear_model = LinearRegression()

In [8]:
linear_model.fit(X_train, Y_train)
print(r2_score(Y_train, linear_model.predict(X_train)) * 100, r2_score(Y_test, linear_model.predict(X_test)) * 100)
print(mean_absolute_error(Y_train, linear_model.predict(X_train)), mean_absolute_error(Y_test, linear_model.predict(X_test)))

18.56792966771653 18.104433392746433
258.41711829869337 258.0645035456383


In [9]:
get_crossval_report(linear_model, X_train, Y_train)

c:\Users\max\anaconda3\Lib\site-packages\plotly\express\_core.py:1979: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  sf: grouped.get_group(s if len(s) > 1 else s[0])


c:\Users\max\anaconda3\Lib\site-packages\plotly\express\_core.py:1979: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



## Градиентный бустинг

In [10]:
catboost_model = CatBoostRegressor(
    iterations=250,
    max_depth=2,
    random_seed=SEED,
    verbose=0
)

In [11]:
catboost_model.fit(X_train, Y_train)
print(r2_score(Y_train, catboost_model.predict(X_train)) * 100, r2_score(Y_test, catboost_model.predict(X_test)) * 100)
print(mean_absolute_error(Y_train, catboost_model.predict(X_train)), mean_absolute_error(Y_test, catboost_model.predict(X_test)))

36.184700616597176 35.63644861962737
227.7998155421168 227.88458756173688


In [12]:
get_crossval_report(catboost_model, X_train, Y_train)

c:\Users\max\anaconda3\Lib\site-packages\plotly\express\_core.py:1979: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



c:\Users\max\anaconda3\Lib\site-packages\plotly\express\_core.py:1979: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



## Финальная модель

In [13]:
catboost_model = CatBoostRegressor(
    iterations=250,
    max_depth=2,
    random_seed=SEED,
    verbose=0
)

catboost_model.fit(X_train, Y_train)

In [14]:
catboost_model.get_feature_importance(prettified=True).head(20)

,Feature Id,Importances
0,MeanPawnLoss1,21.317510
1,ECO,18.800318
2,MeanKnightLoss,15.465427
3,MeanBishopLoss,10.403945
4,OpeningType,9.231663
5,MeanQueenLoss,8.223429
6,MeanRookLoss,5.613544
7,MeanKingLoss2,2.763275
8,ECOType,1.988215
9,MeanFlagPawnLoss1,1.864438


**Custom Metric**

In [15]:
aaa = df_test[["WhiteElo", "BlackElo"]].assign(Prediction=catboost_model.predict(X_test))
# aaa = df_test[["WhiteElo", "BlackElo"]].assign(Prediction=Y_test.mean())
# aaa = df_test[["WhiteElo", "BlackElo"]].assign(Prediction=Y_test.values)


In [ ]:
aaa["Error"] = 0.0

aaa["Error"] = np.min([
    (aaa["Prediction"] - aaa["WhiteElo"]).abs().values ** 2,
    (aaa["Prediction"] - aaa["BlackElo"]).abs().values ** 2
], axis=0)

In [ ]:
is_in = (
    ((aaa["WhiteElo"] <= aaa["Prediction"]) & (aaa["Prediction"] <= aaa["BlackElo"])) |
    ((aaa["BlackElo"] <= aaa["Prediction"]) & (aaa["Prediction"] <= aaa["WhiteElo"]))
)

aaa.loc[is_in, "Error"] = 0

In [ ]:
aaa["Error"].mean()

In [ ]:
1 - 67789/114247